## 1. Importar Librerias

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

## 2. Variables y configuracion

In [2]:
usuario = "SilentStorm63"
listas = ["games", "played", "completed", "backlog", "wishlist", "favorites"]
base_url = f"https://www.backloggd.com/u/{usuario}/games"
resultados = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

## 3. Funcion para scrapear

In [13]:
def scrape_lista(lista):
    page = 1
    while True:
        if lista == "games":
            url = f"{base_url}/?page={page}"
            lista_nombre = "All"
        else:
            url = f"{base_url}/{lista}/?page={page}"
            lista_nombre = lista.capitalize()

        print(f"Scrapeando: {lista_nombre} - Página {page}")
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")
        
        juegos = soup.select(".game-cover")
        # 🔁 Solo continuar si hay juegos válidos
        juegos_validos = [j for j in juegos if j.select_one(".game-text-centered")]

        if not juegos_validos:
            print("🔚 No se encontraron más juegos, deteniendo scraping.")
            break

        for juego in juegos_validos:
            nombre = juego.select_one(".game-text-centered")
            link = juego.select_one("a.cover-link")
            if nombre and link:
                resultados.append({
                    "Juego": nombre.text.strip(),
                    "URL": "https://www.backloggd.com" + link["href"],
                    "Lista": lista_nombre
                })

        page += 1
        time.sleep(1.5)

In [15]:
def scrape_lista(lista):
    page = 1
    juegos_previos = set()

    while True:
        if lista == "games":
            url = f"{base_url}/?page={page}"
            lista_nombre = "All"
        else:
            url = f"{base_url}/{lista}/?page={page}"
            lista_nombre = lista.capitalize()

        print(f"Scrapeando: {lista_nombre} - Página {page}")
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        juegos = soup.select(".game-cover")
        nuevos = 0

        for juego in juegos:
            nombre_elem = juego.select_one(".game-text-centered")
            link_elem = juego.select_one("a.cover-link")

            if nombre_elem and link_elem:
                nombre = nombre_elem.text.strip()
                url_juego = "https://www.backloggd.com" + link_elem["href"]

                if url_juego not in juegos_previos:
                    resultados.append({
                        "Juego": nombre,
                        "URL": url_juego,
                        "Lista": lista_nombre
                    })
                    juegos_previos.add(url_juego)
                    nuevos += 1

        if nuevos == 0:
            print("🔚 No se encontraron más juegos nuevos, deteniendo scraping.")
            break

        page += 1
        time.sleep(1.2)


## 4. Ejecutar Scraping

In [16]:
for lista in listas:
    scrape_lista(lista)

df = pd.DataFrame(resultados).drop_duplicates(subset=["Juego", "Lista"])
print(f"🎮 Juegos encontrados: {len(df)}")

Scrapeando: All - Página 1
Scrapeando: All - Página 2
Scrapeando: All - Página 3
🔚 No se encontraron más juegos nuevos, deteniendo scraping.
Scrapeando: Played - Página 1
Scrapeando: Played - Página 2
Scrapeando: Played - Página 3
🔚 No se encontraron más juegos nuevos, deteniendo scraping.
Scrapeando: Completed - Página 1
Scrapeando: Completed - Página 2
Scrapeando: Completed - Página 3
🔚 No se encontraron más juegos nuevos, deteniendo scraping.
Scrapeando: Backlog - Página 1
Scrapeando: Backlog - Página 2


KeyboardInterrupt: 

## 5. Vista previa

In [20]:
df.head(10)

""


## 6. Guardar en Excel

In [21]:
df.to_excel("backloggd_juegos_basico.xlsx", index=False)
print("✅ Archivo guardado como 'backloggd_juegos_basico.xlsx'")

✅ Archivo guardado como 'backloggd_juegos_basico.xlsx'
